# MetaQuantus Meta-Evaluation — NR + AR Tests

Runs **Noise Resilience (NR)** and **Adversarial Reactivity (AR)** tests
(Hedström et al., TMLR 2024) for every eligible `(model, fae_method, metric)`
triple from the 12-metric vertical slice.

Expected output: `results/meta_evaluation_reliability.csv` with columns
`model, fae_method, metric, nr_score, ar_score, combined_reliability, status`.

**Prerequisites**
- Runtime: **T4 GPU** — Runtime → Change runtime type → T4 GPU
- Google Drive must contain:
  - `thesis/weights/resnet18_isic2017.pth`
  - `thesis/weights/squeezenet_isic2017.pth`
  - `thesis/results/vertical_slice_7fae_12metrics.csv`
  - `thesis/data/` scaffold (images + masks for 12 test images)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

## 2. Clone Repository

In [ ]:
import os
REPO_DIR = '/content/fae-metrics-master-thesis'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/dawkopagh/fae-metrics-master-thesis.git {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')
print('HEAD commit:', end=' ')
!git rev-parse HEAD

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch, scipy
print(f'captum  {captum.__version__}')
print(f'quantus {quantus.__version__}')
print(f'torch   {torch.__version__}')
print(f'scipy   {scipy.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Copy Weights from Drive

In [ ]:
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os
os.makedirs('weights', exist_ok=True)

!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth

print('SHA-256 checksums (actual):')
!sha256sum weights/resnet18_isic2017.pth weights/squeezenet_isic2017.pth
print(f'\nExpected resnet18:   {RESNET_SHA}')
print(f'Expected squeezenet: {SQUEEZENET_SHA}')

## 5. Copy Data Scaffold from Drive

Copies the 12 test images and their masks used in the vertical slice.
Only the test split is needed; no training data.

In [ ]:
import os, shutil

# Copy the test-split images and masks
for split_dir in ['data/images/test', 'data/masks/test']:
    src = f'{DRIVE_THESIS}/{split_dir}'
    if os.path.exists(src):
        os.makedirs(split_dir, exist_ok=True)
        !cp -r {src}/* {split_dir}/
        n = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f'{split_dir}: {n} files')
    else:
        print(f'WARNING: {src} not found on Drive')

print('\nData scaffold ready.')

## 6. Load Vertical Slice CSV from Drive

The `vertical_slice_7fae_12metrics.csv` produced by `colab_vertical_slice.ipynb`
is the input to the pre-screening step.

In [ ]:
import os, shutil, pandas as pd

os.makedirs('results', exist_ok=True)

SLICE_CSV_DRIVE = f'{DRIVE_THESIS}/results/vertical_slice_7fae_12metrics.csv'
SLICE_CSV_LOCAL = 'results/vertical_slice_7fae_12metrics.csv'

shutil.copy(SLICE_CSV_DRIVE, SLICE_CSV_LOCAL)

slice_df = pd.read_csv(SLICE_CSV_LOCAL)
print(f'Loaded: {len(slice_df)} rows')
assert len(slice_df) == 2016, f'Expected 2016 rows, got {len(slice_df)}'
print(f'Models:  {slice_df.model.unique().tolist()}')
print(f'FAE:     {slice_df.fae_method.unique().tolist()}')
print(f'Metrics: {sorted(slice_df.metric.unique().tolist())}')

## 7. Pre-Screening

Determines which `(model, fae_method, metric)` triples are eligible for
meta-evaluation.  Skips triples where valid_fraction < 0.5 (mostly NaN).

In [ ]:
import sys
sys.path.insert(0, '.')

from src.meta_evaluation.metaquantus_wrapper import screen_meta_eval_candidates

candidates = screen_meta_eval_candidates(slice_df, min_valid_fraction=0.5)

total_triples  = len(candidates)
eligible       = int(candidates['run_meta_eval'].sum())
skipped        = total_triples - eligible

print(f'Total candidate triples:  {total_triples}')
print(f'Eligible (run_meta_eval): {eligible}')
print(f'Skipped (low coverage):   {skipped}')
print()

# Show skipped breakdown
skip_df = candidates[~candidates['run_meta_eval']]
if not skip_df.empty:
    print('Skipped triples by metric:')
    print(skip_df.groupby('metric')[['model']].count().rename(columns={'model': 'n_skipped'}).to_string())
    print()

# Runtime estimate: ~50s per triple on T4 (NR: 5 seeds × IG + metric;  AR: 5 levels × metric)
SEC_PER_TRIPLE = 50.0
est_seconds    = eligible * SEC_PER_TRIPLE
est_hours      = est_seconds / 3600

print(f'Estimated runtime: {est_seconds/60:.0f} min  ({est_hours:.1f} h)')
print(f'Budget threshold:  360 min  (6 h T4 session)')

if est_hours > 6.0:
    raise RuntimeError(
        f'Projected runtime {est_hours:.1f} h exceeds 6-hour T4 limit.\n'
        f'Reduce scope: lower n_seeds (currently 5), n_levels (currently 5),\n'
        f'or restrict to fewer metrics/FAE methods before proceeding.'
    )

print('\nRuntime within budget — proceeding.')

## 8. Smoke Test (1 triple at full n_seeds=5, n_levels=5)

Validates the GPU code path before committing to the full run.
If this cell exceeds 120s, do NOT proceed — profile the bottleneck first.

In [ ]:
import time, numpy as np, torch
import quantus

from src.attributions.generate import compute_integrated_gradients
from src.meta_evaluation.metaquantus_wrapper import meta_evaluate_metric
from src.models.classifiers import load_resnet18

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model_r18 = load_resnet18('weights/resnet18_isic2017.pth', device=device)

# Load first test image from the data scaffold
from src.data.isic_dataset import ISIC2017Dataset
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
sample  = dataset[0]
image_np = sample['image'].numpy()
image_t  = sample['image']

with torch.no_grad():
    logits = model_r18(image_t.unsqueeze(0).to(device))
    target = int(logits.argmax(dim=1).item())

attr_t  = compute_integrated_gradients(model_r18, image_t, target, device=device, n_steps=50)
attr_np = attr_t.cpu().numpy()

def _ig_explain_fn(model, inputs, targets, **kwargs):
    attrs = []
    for i in range(inputs.shape[0]):
        img = torch.tensor(inputs[i], dtype=torch.float32)
        a   = compute_integrated_gradients(model, img, int(targets[i]), device=device, n_steps=50)
        attrs.append(a.cpu().numpy())
    return np.stack(attrs, axis=0)

fc_metric = quantus.FaithfulnessCorrelation(
    nr_runs=100, subset_size=224, perturb_baseline='black',
    normalise=True, abs=False, return_aggregate=False, disable_warnings=True,
)

t0 = time.perf_counter()
result = meta_evaluate_metric(
    metric_name='FaithfulnessCorrelation',
    metric_fn=fc_metric,
    model=model_r18,
    images=[image_np],
    attributions=[attr_np],
    targets=[target],
    explain_fn=_ig_explain_fn,
    device=device,
    n_seeds=5,
    n_levels=5,
)
elapsed = time.perf_counter() - t0

nr = result['nr']['nr_score']
ar = result['ar']['ar_score']
print(f'FaithfulnessCorrelation: NR={nr:.3f}, AR={ar:.3f}, combined={result["combined_reliability"]:.3f}')
print(f'Smoke test elapsed: {elapsed:.1f}s')

assert elapsed < 120, f'Smoke test took {elapsed:.0f}s — too slow; profile before proceeding'
assert np.isfinite(nr) and np.isfinite(ar), 'NaN in smoke test scores'
print('\nSmoke test PASSED — GPU path works, proceeding to full run.')

## 9. Full Meta-Evaluation

Iterates over all eligible `(model, fae_method, metric)` triples and writes
results incrementally to `results/meta_evaluation_reliability.csv`.

The output CSV is appended after each `(model, fae)` pair completes so a
session timeout does not lose all work.  Re-running this cell resumes from
the last completed pair.

In [ ]:
import sys, numpy as np, torch, quantus
sys.path.insert(0, '.')

from src.models.classifiers  import load_resnet18, load_squeezenet
from src.data.isic_dataset   import ISIC2017Dataset
from src.attributions.generate import (
    compute_integrated_gradients, compute_saliency, compute_gradcam,
    compute_deep_lift, compute_guided_backprop, compute_lrp, compute_occlusion,
)
from src.models.classifiers  import get_gradcam_target_layer
from src.pipeline            import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import run_meta_evaluation_full

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Models ──────────────────────────────────────────────────────────────────
models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth',   device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}

# ── Test images (same 12 used in the vertical slice) ────────────────────────
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
N_IMAGES = 12
samples  = [dataset[i] for i in range(N_IMAGES)]
images   = [s['image'] for s in samples]            # list of (3, 224, 224) tensors
targets  = [
    int(models['resnet18'](s['image'].unsqueeze(0).to(device)).argmax(dim=1).item())
    for s in samples
]
print(f'Test images loaded: {len(images)}')

# ── FAE explain functions ────────────────────────────────────────────────────
fae_methods = {
    name: _make_explain_func(name, models['resnet18'], 'resnet18', device)
    for name in [
        'integrated_gradients', 'saliency', 'gradcam',
        'deep_lift', 'guided_backprop', 'lrp', 'occlusion',
    ]
}
# Note: _make_explain_func binds model at creation time; the orchestrator
# passes the correct model for each group, so we rebuild per model inside
# a wrapper below.
def _make_fae_dict(model, arch):
    return {
        name: _make_explain_func(name, model, arch, device)
        for name in [
            'integrated_gradients', 'saliency', 'gradcam',
            'deep_lift', 'guided_backprop', 'lrp', 'occlusion',
        ]
    }

# ── Metric instances (production config) ────────────────────────────────────
metric_fns = {
    'faithfulness_correlation': quantus.FaithfulnessCorrelation(
        nr_runs=100, subset_size=224, perturb_baseline='black',
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pixel_flipping': quantus.PixelFlipping(
        features_in_step=224, perturb_baseline='black',
        normalise=True, abs=False, return_aggregate=False,
        return_auc_per_sample=True, disable_warnings=True),
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'avg_sensitivity': quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'relevance_mass_accuracy': quantus.RelevanceMassAccuracy(
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pointing_game': quantus.PointingGame(
        normalise=True, abs=True, return_aggregate=False, disable_warnings=True),
    'sparseness': quantus.Sparseness(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'complexity': quantus.Complexity(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'model_parameter_randomisation': quantus.ModelParameterRandomisation(
        layer_order='top_down', normalise=True, abs=True,
        return_average_correlation=True, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
    'completeness': quantus.Completeness(
        abs=False, normalise=False, perturb_baseline='black',
        return_aggregate=False, disable_warnings=True),
    'non_sensitivity': None,  # disabled — screened out in pre-screening
}
# Remove None entries (screened-out metrics)
metric_fns = {k: v for k, v in metric_fns.items() if v is not None}

# ── Run full meta-evaluation ────────────────────────────────────────────────
# The orchestrator iterates over (model, fae) pairs from screen_meta_eval_candidates.
# It uses the model from the `models` dict and the explain_fn from `fae_methods`.
# For model-specific explain_fns, we pass a combined fae_methods that includes
# both architectures by relying on _make_explain_func's closure.
#
# Implementation note: fae_methods here are bound to resnet18 for the initial
# attribution computation inside run_meta_evaluation_full. The NR test uses
# explain_fn(model, ...) where model is the correct architecture, so the
# _make_explain_func closures correctly re-dispatch on the model argument.
fae_methods_combined = {
    name: _make_explain_func(name, models['resnet18'], 'resnet18', device)
    for name in [
        'integrated_gradients', 'saliency', 'gradcam',
        'deep_lift', 'guided_backprop', 'lrp', 'occlusion',
    ]
}

result_df = run_meta_evaluation_full(
    vertical_slice_df=slice_df,
    models=models,
    metric_fns=metric_fns,
    fae_methods=fae_methods_combined,
    images=images,
    targets=targets,
    device=device,
    n_seeds=5,
    n_levels=5,
    output_csv='results/meta_evaluation_reliability.csv',
    progress_log='results/meta_eval_progress.log',
)

print(f'\nCompleted: {len(result_df)} triples')
print(result_df['status'].value_counts().to_string())

## 10. Post-Processing

Summarises NR, AR, and combined reliability scores per metric.

In [ ]:
import pandas as pd

rel_df = pd.read_csv('results/meta_evaluation_reliability.csv')

completed = rel_df[rel_df['status'] == 'completed']

print(f'Total triples in CSV:  {len(rel_df)}')
print(f'Completed:             {len(completed)}')
print(f'Skipped (NaN):         {(rel_df["status"]=="skipped_nan").sum()}')
print(f'Failed:                {(rel_df["status"]=="failed").sum()}')
print()

# Mean NR, AR, combined per metric across (model, fae) combinations
summary = (
    completed
    .groupby('metric')[['nr_score', 'ar_score', 'combined_reliability']]
    .mean()
    .round(3)
    .sort_values('combined_reliability', ascending=False)
)
print('=== Mean reliability per metric (sorted by combined) ===')
print(summary.to_string())

print()
print('=== Least reliable metrics (combined < 0.5) ===')
low_rel = summary[summary['combined_reliability'] < 0.5]
if low_rel.empty:
    print('  (none below 0.5)')
else:
    print(low_rel.to_string())

## 11. Copy Results to Drive

In [ ]:
import shutil, os

results_to_save = [
    'results/meta_evaluation_reliability.csv',
    'results/meta_eval_progress.log',
]

for src in results_to_save:
    if os.path.exists(src):
        dest = f'{DRIVE_THESIS}/{src}'
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(src, dest)
        print(f'Saved {src} \u2192 {dest}')
    else:
        print(f'WARNING: {src} not found, skipping.')

print('\nDone. Download results/ to local fae-metrics-master-thesis/ before the next session.')